In [1]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import roc_auc_score, brier_score_loss
from sklearn.calibration import calibration_curve, IsotonicRegression
from sklearn.model_selection import train_test_split

# Calibration Breakthrough: What Actually Improved

This notebook demonstrates a key concept in clinical prediction models:

- **AUROC changed minimally** (ranking nearly unchanged).
- **Brier and ECE improved sharply** (probability trustworthiness improved).
- **Clinical utility increases** when output probabilities become reliable risk estimates.

In [2]:
# Set plot style
sns.set_theme(style="whitegrid")
np.random.seed(42)

# ==========================================
# 1. Generate Synthetic Data
# ==========================================
# True probabilities (risk of disease)
n_samples = 10000
true_probs = np.random.beta(2, 8, n_samples)  # Skewed towards lower risks

# True labels
y_true = np.random.binomial(1, true_probs)

# Simulate an uncalibrated model: 
# It preserves the ranking (monotonic) but scales poorly 
# (e.g., predicted probabilities are squashed between 0.3 and 0.7)
y_pred_uncal = 0.3 + 0.4 * true_probs

# Add some random noise to uncalibrated predictions so they are not perfectly monotonic
y_pred_uncal += np.random.normal(0, 0.05, n_samples)
y_pred_uncal = np.clip(y_pred_uncal, 0.01, 0.99)

# Split into train (for calibration) and test sets
X_train, X_test, y_train, y_test = train_test_split(y_pred_uncal, y_true, test_size=0.5, random_state=42)

# Calibrate using Isotonic Regression
iso = IsotonicRegression(out_of_bounds='clip')
iso.fit(X_train, y_train)

# Get predictions on test set
y_pred_uncal_test = X_test
y_pred_cal_test = iso.predict(X_test)

## 1. Minimal Change in AUROC, Sharp Improvement in Brier & ECE

In [3]:
def expected_calibration_error(y_true, y_prob, n_bins=10):
    prob_true, prob_pred = calibration_curve(y_true, y_prob, n_bins=n_bins)
    bin_edges = np.linspace(0., 1., n_bins + 1)
    bin_indices = np.digitize(y_prob, bin_edges) - 1
    
    ece = 0
    for i in range(n_bins):
        mask = bin_indices == i
        if np.any(mask):
            bin_size = np.sum(mask)
            ece += (bin_size / len(y_prob)) * np.abs(prob_true[i] - prob_pred[i])
    return ece

# Calculate metrics
auroc_uncal = roc_auc_score(y_test, y_pred_uncal_test)
auroc_cal = roc_auc_score(y_test, y_pred_cal_test)

brier_uncal = brier_score_loss(y_test, y_pred_uncal_test)
brier_cal = brier_score_loss(y_test, y_pred_cal_test)

ece_uncal = expected_calibration_error(y_test, y_pred_uncal_test)
ece_cal = expected_calibration_error(y_test, y_pred_cal_test)

print("=== Model Performance Comparison ===")
print(f"AUROC (Ranking)      | Uncalibrated: {auroc_uncal:.4f} -> Calibrated: {auroc_cal:.4f}  (Minimal Change)")
print(f"Brier Score          | Uncalibrated: {brier_uncal:.4f} -> Calibrated: {brier_cal:.4f}  (Improved Sharply)")
print(f"ECE (Calibration)    | Uncalibrated: {ece_uncal:.4f} -> Calibrated: {ece_cal:.4f}  (Improved Sharply)")

IndexError: index 6 is out of bounds for axis 0 with size 6

## 2. Visualizing Calibration Improvements

In [ ]:
plt.figure(figsize=(8, 6))

prob_true_uncal, prob_pred_uncal = calibration_curve(y_test, y_pred_uncal_test, n_bins=10)
prob_true_cal, prob_pred_cal = calibration_curve(y_test, y_pred_cal_test, n_bins=10)

plt.plot([0, 1], [0, 1], linestyle='--', color='black', label='Perfectly Calibrated')
plt.plot(prob_pred_uncal, prob_true_uncal, marker='o', label=f'Uncalibrated (ECE={ece_uncal:.3f})')
plt.plot(prob_pred_cal, prob_true_cal, marker='s', label=f'Calibrated (ECE={ece_cal:.3f})')

plt.xlabel('Mean Predicted Probability')
plt.ylabel('Fraction of Positives')
plt.title('Calibration Curve (Reliability Diagram)')
plt.legend()
plt.show()

## 3. Clinical Utility Increases (Decision Curve Analysis)

Net benefit compares the benefits of true positives against the harms of false positives, weighted by a threshold probability representing how much the clinician weighs false positives relative to missed true positives.

In [ ]:
def calculate_net_benefit(y_true, y_prob, thresholds):
    net_benefit = []
    for t in thresholds:
        tp = np.sum((y_prob >= t) & (y_true == 1))
        fp = np.sum((y_prob >= t) & (y_true == 0))
        n = len(y_true)
        
        if t == 1:
            nb = 0
        else:
            nb = (tp / n) - (fp / n) * (t / (1 - t))
        net_benefit.append(nb)
    return np.array(net_benefit)

# Relevant clinical thresholds
thresholds = np.linspace(0.01, 0.40, 100) 

nb_uncal = calculate_net_benefit(y_test, y_pred_uncal_test, thresholds)
nb_cal = calculate_net_benefit(y_test, y_pred_cal_test, thresholds)

# Baseline strategies
nb_all = calculate_net_benefit(y_test, np.ones_like(y_test), thresholds)
nb_none = np.zeros_like(thresholds)

plt.figure(figsize=(10, 6))
plt.plot(thresholds, nb_all, label="Treat All", linestyle=':', color='gray')
plt.plot(thresholds, nb_none, label="Treat None", linestyle='-.', color='black')
plt.plot(thresholds, nb_uncal, label="Uncalibrated Model", linewidth=2)
plt.plot(thresholds, nb_cal, label="Calibrated Model", linewidth=2)

plt.ylim(-0.02, max(nb_all) + 0.05)
plt.xlim(0, 0.4)
plt.xlabel("Threshold Probability (Risk Tolerance)")
plt.ylabel("Net Benefit")
plt.title("Decision Curve Analysis: Clinical Utility Improvement")
plt.legend()
plt.show()

### Conclusion
As shown in the curves above, **calibrating the model barely changed the ranking (AUROC)**. However, because the probabilities became trustworthy (reflected by lower Brier and ECE scores), the **calibrated model provides a much higher Net Benefit across clinically relevant thresholds**. This demonstrates that *clinical utility increases when output probabilities become reliable risk estimates.*